# 6.6 邊界標記怎麼保持完整？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**用戶打字「<assistant>」會自動變成assistant角色嗎？

角色與模態邊界要有獨立ID。內容文字裡出現相同拼寫，不能偷偷升級成控制token；結構化role字段由資料工具決定。

**把直覺寫成數學：**本文ByteTokenizer用0–7作專用邊界，原文bytes加8；兩者ID集合不重疊。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import ByteTokenizer, SPECIALS

tok = ByteTokenizer()
print(list(enumerate(SPECIALS)))
literal = tok.encode("<assistant>")
print(literal)
assert tok.assistant_id not in literal

**如何讀結果：**完整BPE版本使用registered added tokens，但render仍須避免把不可信文字當角色。

**只改一件事：**只在user文字放入圖片標記拼寫，檢查仍是文字。
